In [ ]:
import json
import numpy as np
import pandas as pd
import arviz as az

from emu_renewal.inputs import get_world_shp
from emu_renewal.constants import DATA_PATH, FULL_RUN, OXCGRT_COLMAP, MOB_LOCATION_NAME_MAP
from emu_renewal.utils import get_analysis_paths
from emu_renewal.plotting import plot_param_map, plot_best_policy

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
policy_codes = OXCGRT_COLMAP["custom"]

records = []
for iso3, analyses in analysis_paths.items():
    analysis_path = analyses["oxcgrt_floored"]
    idata = az.from_netcdf(analysis_path / "idata_filtered.nc")
    medians = idata.posterior["ts_weights"].median(dim=("chain", "draw"))
    scale_floor = float(idata.posterior["scale_floor"].median(dim=("chain", "draw")))
    scale_exp = float(idata.posterior["scale_exp"].median(dim=("chain", "draw")))
    best_policy = int(medians.argmax().item())
    scale_factor = (1.0 - scale_floor) / float(np.sum(medians))
    row = {
        "ISO_A3": iso3,
        "best_policy": best_policy,
        "floor": scale_floor,
        "floor_complement": 1.0 - scale_floor,
        "scale_exp": scale_exp,
        "best_name": policy_codes[int(best_policy)]
    }
    for k, v in enumerate(medians):
        row[f"pol_{k}"] = float(v) * scale_factor
    records.append(row)
data_df = pd.DataFrame.from_records(records)

world = world.merge(data_df, on="ISO_A3", how="left")
missing = world[world["best_policy"].isna()]
exclude = world[(world["floor"] > 0.75) | (world["scale_exp"] < 0.75)]

## Complement of the scale floor

In [ ]:
plot_param_map(world, "floor_complement", 1.0)

## Scale exponent

In [ ]:
plot_param_map(world, "scale_exp", 2.0)

## Best policy

In [ ]:
plot_best_policy(world, missing, exclude)

In [ ]:
for a in range(len(medians)):
    policy = MOB_LOCATION_NAME_MAP[policy_codes[a]]
    plot_param_map(world, f"pol_{a}", 0.14, excluded=exclude, title=policy)
